# Hands-on Exercise 2a — Scaling a Distributed ML Job with Kubernetes
### AI Operations (AIOps) — Module 3, Lecture 2a | ~90–120 minutes

**Referenced in:** *Module3_Slides_2a_Kubernetes.pptx*

**Objective:** stand up a local Kubernetes cluster with minikube, package a CPU-only distributed
hyperparameter search as a container, run it as a Kubernetes **Indexed Job** first across
multiple pods on **one node**, then across multiple pods on **three nodes**, and empirically
measure how wall-clock time scales with parallelism.

**Why this workload:** a hyperparameter search — training the same model many times with
different settings — is *embarrassingly parallel*: each run is fully independent, needs no
communication with the others, and is exactly the kind of workload Kubernetes Jobs are built
for. It requires **no GPU** and finishes in seconds per run, so the whole exercise runs
comfortably on a laptop.

**Deliverable:**
- Screenshots/output showing pods running concurrently on one node (`kubectl get pods -o wide`)
- Screenshots/output showing pods spread across 3 nodes
- The collected results table (12 hyperparameter combinations, with accuracy and which pod/node ran each)
- The scalability plot (parallelism vs. wall-clock time) from `scalability_test.py`

> **A note on realism:** this notebook writes real files to disk and gives you the exact commands
> to run. The `kubectl`/`minikube` commands themselves must be run in a **terminal**, not inside
> a notebook cell with `!`, because several of them (cluster start, multi-node reconfiguration)
> take minutes and are much easier to monitor and troubleshoot interactively. Cells that call the
> Kubernetes Python API directly (for results collection and the scalability sweep) **do** run
> inline, once your cluster is up.

## Part 0 — Install minikube and kubectl

Skip this part if you already completed the setup during Lecture 1 (Docker) or a prior session.

### Linux
```bash
curl -LO https://storage.googleapis.com/minikube/releases/latest/minikube-linux-amd64
sudo install minikube-linux-amd64 /usr/local/bin/minikube

curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
sudo install kubectl /usr/local/bin/kubectl
```

### Verify the install
```bash
minikube version
kubectl version --client
```

> minikube needs a container/VM driver. Docker Desktop (installed in Lecture 1) works as the
> driver on all three platforms — no separate hypervisor needed:
> `minikube config set driver docker`

## Part 1 — Start a single-node cluster

We start small (1 node, 4 CPUs) to demonstrate multi-**pod** parallelism on a single node first,
before scaling out to multiple nodes in Part 6.

In [ ]:
# Run this in a TERMINAL (not this notebook cell) -- it takes 1-3 minutes:
#
#   minikube start --cpus 4 --memory 4096 --driver=docker
#
# Then verify from here:
import subprocess

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("kubectl cluster-info")
sh("kubectl get nodes -o wide")

_Expected output:_ one node, `STATUS: Ready`. If `kubectl cluster-info` fails, minikube likely
isn't running yet — go back to the terminal and confirm `minikube start` finished successfully.

## Part 2 — Generate the dataset

A small, deterministic synthetic classification dataset — the same one used again in Lecture 2b.

In [ ]:
%%writefile generate_dataset.py
"""
generate_dataset.py — AI Operations (AIOps), Module 3 Lecture 2a
Generates a small, deterministic synthetic classification dataset.
"""
import argparse
import pandas as pd
from sklearn.datasets import make_classification


def generate(n_samples=6000, n_features=20, n_informative=12, n_classes=2, random_state=42):
    X, y = make_classification(
        n_samples=n_samples, n_features=n_features, n_informative=n_informative,
        n_redundant=4, n_classes=n_classes, class_sep=1.1, random_state=random_state,
    )
    columns = [f"feature_{i:02d}" for i in range(n_features)]
    df = pd.DataFrame(X, columns=columns)
    df["label"] = y
    return df


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="ml_job_dataset.csv")
    parser.add_argument("--n-samples", type=int, default=6000)
    parser.add_argument("--n-features", type=int, default=20)
    parser.add_argument("--random-state", type=int, default=42)
    args = parser.parse_args()

    df = generate(n_samples=args.n_samples, n_features=args.n_features, random_state=args.random_state)
    df.to_csv(args.out, index=False)
    print(f"Wrote {len(df):,} rows x {df.shape[1]} columns to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, os
os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "generate_dataset.py", "--out", "data/ml_job_dataset.csv"], check=True)
print(os.path.getsize("data/ml_job_dataset.csv"), "bytes")

## Part 3 — Write the training worker script

This is the entry point every Job pod runs. It reads `JOB_COMPLETION_INDEX` (set automatically by Kubernetes for **Indexed Jobs**) to pick one point from a 12-combination hyperparameter grid.

In [ ]:
%%writefile train_worker.py
"""
train_worker.py — AI Operations (AIOps), Module 3 Lecture 2a
Entry point for each pod of the Kubernetes Indexed Job.
"""
import itertools, json, os, socket, time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

N_ESTIMATORS_GRID = [50, 100, 150, 200]
MAX_DEPTH_GRID = [4, 8, 12]
GRID = list(itertools.product(N_ESTIMATORS_GRID, MAX_DEPTH_GRID))


def main():
    completion_index = int(os.environ.get("JOB_COMPLETION_INDEX", "0"))
    n_estimators, max_depth = GRID[completion_index % len(GRID)]

    data_path = os.environ.get("DATA_PATH", "/app/data/ml_job_dataset.csv")
    pod_name = os.environ.get("POD_NAME", socket.gethostname())
    node_name = os.environ.get("NODE_NAME", "unknown")

    print(f"[worker {completion_index}] pod={pod_name} node={node_name} "
          f"n_estimators={n_estimators} max_depth={max_depth}", flush=True)

    df = pd.read_csv(data_path)
    X = df.drop(columns=["label"])
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    t0 = time.time()
    # n_jobs=1 is DELIBERATE: each pod should use ~1 CPU core, so the parallelism story is
    # about KUBERNETES scheduling many pods, not a single RandomForest fanning out internally.
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=1)
    model.fit(X_train, y_train)
    train_seconds = time.time() - t0

    preds = model.predict(X_test)
    result = {
        "completion_index": completion_index, "n_estimators": n_estimators, "max_depth": max_depth,
        "accuracy": round(accuracy_score(y_test, preds), 4),
        "f1_score": round(f1_score(y_test, preds), 4),
        "train_seconds": round(train_seconds, 3),
        "pod_name": pod_name, "node_name": node_name,
    }
    print("RESULT_JSON:" + json.dumps(result), flush=True)


if __name__ == "__main__":
    main()


**Quick local sanity check** — simulate a couple of completion indices without Kubernetes at all, to confirm the script itself is correct before we containerize it:

In [ ]:
import subprocess, os

env = os.environ.copy()
env.update({"DATA_PATH": "data/ml_job_dataset.csv", "POD_NAME": "local-test", "NODE_NAME": "local"})

for idx in [0, 5, 11]:
    env["JOB_COMPLETION_INDEX"] = str(idx)
    result = subprocess.run(["python3", "train_worker.py"], env=env, capture_output=True, text=True)
    print(result.stdout)

## Part 4 — Containerize the worker

Same multi-stage pattern from Lecture 1. The dataset is baked into the image, so no shared/networked storage is needed on any node.

In [ ]:
%%writefile requirements.txt
pandas
scikit-learn


In [ ]:
%%writefile Dockerfile.job
# --- Stage 1: build ---
FROM python:3.10 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/build/deps -r requirements.txt

# --- Stage 2: minimal runtime ---
FROM python:3.10-slim
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY train_worker.py .
COPY data/ml_job_dataset.csv ./data/ml_job_dataset.csv
CMD ["python", "train_worker.py"]


**Build the image directly inside minikube's Docker environment** — this is the key step that
makes the image available to the cluster without needing a registry. Run in a terminal:

```bash
minikube image build -t hp-search-worker:latest -f Dockerfile.job .
```

Verify it landed:
```bash
minikube image ls | grep hp-search-worker
```

## Part 5 — Run the single-node, multi-pod Job

`parallelism: 4` means up to 4 pods run **concurrently**; `completions: 12` means 12 total hyperparameter combinations get run.

In [ ]:
%%writefile job-single-node.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: hp-search-job
  labels:
    app: hp-search
spec:
  completions: 12
  parallelism: 4
  completionMode: Indexed
  backoffLimit: 4
  activeDeadlineSeconds: 600
  template:
    metadata:
      labels:
        app: hp-search
    spec:
      restartPolicy: Never
      containers:
      - name: worker
        image: hp-search-worker:latest
        imagePullPolicy: IfNotPresent
        env:
        - name: DATA_PATH
          value: "/app/data/ml_job_dataset.csv"
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "900m"
            memory: "256Mi"
          limits:
            cpu: "1000m"
            memory: "512Mi"


Apply it and watch pods start concurrently — run in a terminal so you can watch it live:

```bash
kubectl apply -f job-single-node.yaml
kubectl get pods -o wide -w
```

Press Ctrl-C once all 12 pods show `Completed`. You should see **up to 4 pods `Running`
simultaneously**, all on the same (only) node, before the rest queue up as earlier ones finish.

### Collect the results

Instead of a shared volume, we pull results back via each pod's **logs**, using the Kubernetes API directly:

In [ ]:
%%writefile collect_results.py
"""
collect_results.py — AI Operations (AIOps), Module 3 Lecture 2a
Collects RESULT_JSON lines from every pod of a completed Job via the Kubernetes API.
"""
import argparse, json, re, sys
import pandas as pd
from kubernetes import client, config

RESULT_LINE_RE = re.compile(r"RESULT_JSON:(\{.*\})")


def load_kube_config():
    try:
        config.load_kube_config()
    except Exception:
        config.load_incluster_config()


def collect(job_name, namespace="default"):
    load_kube_config()
    v1 = client.CoreV1Api()
    pods = v1.list_namespaced_pod(namespace=namespace, label_selector=f"job-name={job_name}")
    if not pods.items:
        print(f"No pods found for job '{job_name}'.", file=sys.stderr)
        return pd.DataFrame()

    rows = []
    for pod in pods.items:
        pod_name = pod.metadata.name
        try:
            logs = v1.read_namespaced_pod_log(name=pod_name, namespace=namespace)
        except client.exceptions.ApiException as e:
            print(f"  Could not read logs for {pod_name}: {e.reason}", file=sys.stderr)
            continue
        match = RESULT_LINE_RE.search(logs)
        if not match:
            print(f"  No RESULT_JSON line in {pod_name} yet.", file=sys.stderr)
            continue
        result = json.loads(match.group(1))
        result["k8s_pod_phase"] = pod.status.phase
        rows.append(result)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("completion_index").reset_index(drop=True)
    return df


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--job-name", required=True)
    parser.add_argument("--namespace", default="default")
    parser.add_argument("--out", default=None)
    args = parser.parse_args()

    df = collect(args.job_name, args.namespace)
    if df.empty:
        print("No results collected.")
    else:
        pd.set_option("display.width", 120)
        print(df.to_string(index=False))
        print("\nNodes that participated:", sorted(df["node_name"].unique()))
        if args.out:
            df.to_csv(args.out, index=False)


In [ ]:
from collect_results import collect

results_df = collect("hp-search-job")
results_df

_Expected:_ 12 rows, one per hyperparameter combination, each showing which `pod_name` ran it.
Since this is the single-node run, `node_name` should be the SAME for every row.

In [ ]:
print("Distinct nodes used:", results_df["node_name"].unique())
print("Distinct pods used:", results_df["pod_name"].nunique())
best = results_df.loc[results_df["accuracy"].idxmax()]
print(f"\nBest hyperparameters: n_estimators={best.n_estimators}, max_depth={best.max_depth}, accuracy={best.accuracy}")

## Part 6 — Scale to a 3-node cluster

Now we make the multi-node story real. minikube can add nodes to a running cluster.

Run in a terminal:

```bash
minikube node add
minikube node add
kubectl get nodes -o wide
```

You should now see **3 nodes**, all `Ready`. (Alternatively, start fresh with
`minikube start --nodes 3 --cpus 2 --memory 4096 --driver=docker` if you'd rather not add nodes to the existing cluster.)

> From v1.29.0 onwards, the default container-runtime changed from `docker` to `containerd`.

**Important:** the worker image needs to be loaded onto EVERY node, not just the control-plane
node it was built on:

```bash
minikube image load hp-search-worker:latest
```

Verify on each node (Runtime=docker):
```bash
minikube ssh --node minikube docker images
minikube ssh --node minikube-m02 docker images
minikube ssh --node minikube-m03 docker images
```

Verify on each node (Runtime=containerd):
```bash
minikube ssh --node minikube "sudo ctr -n k8s.io i ls | grep search"
minikube ssh --node minikube-m02 "sudo ctr -n k8s.io i ls | grep search"
minikube ssh --node minikube-m03 "sudo ctr -n k8s.io i ls | grep search"
```

In [ ]:
sh("kubectl get nodes -o wide")

### Run the multi-node Job

Same workload, but `parallelism: 6` with a `topologySpreadConstraints` hint that pushes the scheduler to spread pods across all 3 nodes rather than packing them onto one.

In [ ]:
%%writefile job-multi-node.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: hp-search-job-multinode
  labels:
    app: hp-search
spec:
  completions: 12
  parallelism: 6
  completionMode: Indexed
  backoffLimit: 4
  activeDeadlineSeconds: 600
  template:
    metadata:
      labels:
        app: hp-search
    spec:
      restartPolicy: Never
      topologySpreadConstraints:
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: ScheduleAnyway
        labelSelector:
          matchLabels:
            app: hp-search
      containers:
      - name: worker
        image: hp-search-worker:latest
        imagePullPolicy: IfNotPresent
        env:
        - name: DATA_PATH
          value: "/app/data/ml_job_dataset.csv"
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "900m"
            memory: "256Mi"
          limits:
            cpu: "1000m"
            memory: "512Mi"


Apply and watch — this time pay close attention to the `NODE` column:

```bash
kubectl apply -f job-multi-node.yaml
kubectl get pods -o wide -w
```

In [ ]:
results_multinode_df = collect("hp-search-job-multinode")
results_multinode_df

In [ ]:
print("Distinct nodes used:", sorted(results_multinode_df["node_name"].unique()))
print("Distinct pods used:", results_multinode_df["pod_name"].nunique())
assert results_multinode_df["node_name"].nunique() > 1, "Expected pods to be spread across multiple nodes!"
print("\nConfirmed: work was distributed across multiple nodes.")

## Part 7 — Test scalability: does more parallelism actually help?

This is the direct, empirical answer to "does Kubernetes actually scale this job?" We resubmit
the SAME total work (`completions=12`) at increasing `parallelism`, and measure wall-clock time
for each.

In [ ]:
%%writefile scalability_test.py
"""
scalability_test.py — AI Operations (AIOps), Module 3 Lecture 2a
Sweeps Job parallelism and measures wall-clock completion time.
"""
import argparse, time
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from kubernetes import client, config


def load_kube_config():
    try:
        config.load_kube_config()
    except Exception:
        config.load_incluster_config()


def run_job_at_parallelism(parallelism, completions, namespace, job_template_path, timeout=600):
    batch_v1 = client.BatchV1Api()
    with open(job_template_path) as f:
        job_manifest = yaml.safe_load(f)

    job_name = f"hp-search-scaletest-p{parallelism}"
    job_manifest["metadata"]["name"] = job_name
    job_manifest["spec"]["parallelism"] = parallelism
    job_manifest["spec"]["completions"] = completions

    try:
        batch_v1.delete_namespaced_job(job_name, namespace, propagation_policy="Background")
        time.sleep(5)
    except client.exceptions.ApiException as e:
        if e.status != 404:
            raise

    start = time.time()
    batch_v1.create_namespaced_job(namespace, job_manifest)

    while True:
        status = batch_v1.read_namespaced_job_status(job_name, namespace).status
        if status.succeeded and status.succeeded >= completions:
            break
        if status.failed and status.failed > 0:
            raise RuntimeError(f"Job {job_name} reported {status.failed} failed pod(s).")
        if time.time() - start > timeout:
            raise TimeoutError(f"Job {job_name} did not complete within {timeout}s.")
        time.sleep(2)

    return time.time() - start


def run_scalability_sweep(parallelism_values, completions, namespace, job_template_path):
    load_kube_config()
    rows = []
    for p in parallelism_values:
        print(f"Running with parallelism={p} (completions={completions}) ...")
        elapsed = run_job_at_parallelism(p, completions, namespace, job_template_path)
        print(f"  parallelism={p} -> {elapsed:.1f}s")
        rows.append({"parallelism": p, "elapsed_seconds": round(elapsed, 2)})
    return pd.DataFrame(rows)


def plot_scalability(df, out_path):
    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(df["parallelism"], df["elapsed_seconds"], marker="o", color="tab:blue", label="Wall-clock time")
    ax1.set_xlabel("Parallelism (concurrent pods)")
    ax1.set_ylabel("Wall-clock time (seconds)", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.grid(True, alpha=0.3)
    ax1.set_title("Kubernetes Job Scalability: Parallelism vs. Wall-Clock Time")

    baseline = df["elapsed_seconds"].iloc[0]
    speedup = baseline / df["elapsed_seconds"]
    ax2 = ax1.twinx()
    ax2.plot(df["parallelism"], speedup, marker="s", color="tab:orange", label="Speedup vs. parallelism=1")
    ax2.set_ylabel("Speedup (x)", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    print(f"Saved plot to {out_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--parallelism-values", type=int, nargs="+", default=[1, 2, 4])
    parser.add_argument("--completions", type=int, default=12)
    parser.add_argument("--namespace", default="default")
    parser.add_argument("--job-template", default="job-single-node.yaml")
    parser.add_argument("--out-csv", default="scalability_results.csv")
    parser.add_argument("--out-plot", default="scalability_plot.png")
    args = parser.parse_args()

    df = run_scalability_sweep(args.parallelism_values, args.completions, args.namespace, args.job_template)
    df.to_csv(args.out_csv, index=False)
    print(df.to_string(index=False))
    plot_scalability(df, args.out_plot)


Run the sweep on the single-node cluster (parallelism limited by 4 CPUs) — this takes a few minutes:

In [ ]:
from scalability_test import run_scalability_sweep, plot_scalability

sweep_df = run_scalability_sweep(
    parallelism_values=[1, 4, 6],
    completions=12,
    namespace="default",
    job_template_path="job-single-node.yaml",
)
sweep_df

In [ ]:
plot_scalability(sweep_df, "scalability_plot.png")

#from IPython.display import Image
#Image("scalability_plot.png")

_Discussion:_ is the speedup at parallelism=4 close to 4x? If not, what CPU/scheduling overhead
might explain the gap? (Hint: recall the per-worker training time varies from ~0.3s to ~2.8s
across the grid — with only 4 concurrent slots and 12 total jobs, the LAST batch to finish
determines wall-clock time, so uneven work sizes matter.)

**Optional extension:** rerun the same sweep against `job-multi-node.yaml` with
`parallelism_values=[2, 4, 6]` on your 3-node cluster, and compare the achievable parallelism
ceiling against the single-node run.

## ✅ Deliverable Checklist
- [ ] Single-node run: `kubectl get pods -o wide` output showing up to 4 pods `Running` concurrently
- [ ] Collected results table (12 rows) from the single-node run
- [ ] Multi-node run: `kubectl get pods -o wide` output showing pods spread across 3 distinct nodes
- [ ] Collected results table (12 rows) from the multi-node run, with `node_name` showing ≥ 2 distinct values
- [ ] The scalability plot (parallelism vs. wall-clock time and speedup)
- [ ] A one-paragraph discussion of what limited the observed speedup

*Next: proceed to `Lecture2b_KServe_Scaling_Exercise.ipynb`.*